# S05 · KV eviction: StreamingLLM, H2O, SnapKV

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 11. Sources: github.com/mit-han-lab/streaming-llm (7.3k stars, MIT); arXiv:2306.14048; arXiv:2404.14469; NVIDIA/kvpress.

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Compare three eviction rules at equal token budgets on a context with a few important tokens.

## Theory and equations
Error: $\|\operatorname{attn}(q, K_S, V_S) - \operatorname{attn}(q, K, V)\| / \|\operatorname{attn}(q,K,V)\|$ for the kept set $S$.

## Diagram
```
StreamingLLM: [sinks]..........................................[recent window]
H2O:          [heavy hitters by accumulated attention]          [recent window]
SnapKV:       [top tokens voted by the last w prompt queries, pooled]  [obs window]
```

## Implementation

In [ ]:
from inference_lab import eviction as E
k, v, d, pos = E.needle_context()
q_obs, q_future = E.queries_toward(d, 32, seed=5), E.queries_toward(d, 16, seed=6)
hist = E.attention_probs(q_obs, k)
print("needle positions:", pos.tolist())

## Experiment

In [ ]:
budgets = [64, 128, 256, 512]
res = {"StreamingLLM": [], "H2O": [], "SnapKV": []}
for B in budgets:
    res["StreamingLLM"].append(E.output_error(q_future, k, v, E.streaming(1024, B)))
    res["H2O"].append(E.output_error(q_future, k, v, E.h2o(hist, B, B // 2)))
    res["SnapKV"].append(E.output_error(q_future, k, v, E.snapkv(q_obs, k, B)))
for (name, ys), col in zip(res.items(), [C["waste"], C["memory"], C["compute"]]):
    plt.plot(budgets, ys, "o-", color=col, label=name)
    print(name, [round(y, 3) for y in ys])
plt.xscale("log", base=2); plt.yscale("log"); plt.xlabel("kept tokens (of 1024)"); plt.ylabel("relative output error")
plt.legend(); plt.show()

## Results: when attention is diffuse

In [ ]:
k2, v2, d2, _ = E.needle_context(strength=4.0)
q2 = E.queries_toward(d2, 16, seed=6)
kept = (E.streaming(1024, 256), E.snapkv(E.queries_toward(d2, 32, seed=5), k2, 256))
print("diffuse, budget 256:", [round(E.output_error(q2, k2, v2, s), 3) for s in kept])

## Interpretation
Content-aware rules find the needles; the content-blind window cannot. When attention is
diffuse no small budget works, which is why eviction quality is task-dependent.

## Limitations
One head, synthetic keys, one query distribution.

## Conclusion
Eviction is cheap and lossy; its quality depends on how peaked attention is for the task.